# Catchment analysis for quantifying how NbS reduce river flood risk

### Step 0: Import packages to work with, set up folder pathways and project to Jamaica's grid coordinates

In [ ]:
import geopandas as gpd
from pathlib import Path
import matplotlib as mpl

import matplotlib.pyplot as plt
import pandas as pd
import Robyn_catchment_analysis
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Inputs")
output_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data")
output_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Outputs")

jamaica_metric_grid_crs = "EPSG:3448"

### Step 1: Read in hydrobasins file, then read in Jamaica boundary and clip hydrobasins to Jamaica

In [ ]:
original_hydrobasins_path = base_path / "Hydrobasins_12/hybas_na_lev12_v1c.shp" 
original_hydrobasins = gpd.read_file(original_hydrobasins_path)

In [ ]:
#print(f"Number of unique HYBAS_ID values: {unique_hybas_ids_original}")
unique_hybas_ids_unclipped = original_hydrobasins['HYBAS_ID'].nunique()
print(f"Number of unique HYBAS_ID values: {unique_hybas_ids_unclipped}")

In [ ]:
jamaica_boundary_path = base_path / "Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

In [ ]:
# Reproject HydroBASINS to match Jamaica's CRS if necessary
reprojected_hydrobasins = original_hydrobasins.to_crs(jamaica_boundary.crs)
print(original_hydrobasins.crs)
print(reprojected_hydrobasins.crs)

In [ ]:
hydrobasins_clipped = gpd.clip(reprojected_hydrobasins, jamaica_boundary)
hydrobasins_clipped.plot()

In [ ]:
# Check that the hydrobasins and Jamaica boundary are on the right coordinate system and that they are bounded the same (Jamaica bounds)
print("Hydrobasins_clipped CRS:", hydrobasins_clipped.crs)
print("Jamaica Boundary CRS:", jamaica_boundary.crs)

print("Hydrobasins_clipped bounds:", hydrobasins_clipped.total_bounds)
print("Jamaica boundary bounds:", jamaica_boundary.total_bounds)

In [ ]:
output_shapefile = output_path / "HydroBASINS_Level12_Clipped_Jamaica.shp"
hydrobasins_clipped.to_file(output_shapefile)

In [ ]:
# Read the saved shapefile into a GeoDataFrame
hydrobasins_clipped_saved = gpd.read_file(output_shapefile)

In [ ]:
# Count the unique values in the 'HYBAS_ID' column to check how many rows there should be
unique_hybas_ids_original = hydrobasins_clipped_saved['HYBAS_ID'].nunique()

# Print the result
print(f"Number of unique HYBAS_ID values: {unique_hybas_ids_original}")

In [ ]:
print("hydrobasins_clipped_saved CRS:", hydrobasins_clipped.crs)
print("hydrobasins_clipped_saved bounds:", hydrobasins_clipped.total_bounds)

### Step 2: Read in land use for Jamaica and intersect it with the clipped and re-projected hydrobasins file

In [ ]:
land_use = gpd.read_file(base_path / "2013_landuse_LandCover.shp")
print(land_use.crs)
land_use.plot()

In [ ]:
print(land_use.columns.tolist())

In [ ]:
# 2. Compute area_km2 if you haven’t already
land_use["area_km2"] = land_use.geometry.area / 1e6

# 3. Define classes and subset
classes = [
    "Fields and Secondary Forest",
    "Bamboo and Secondary Forest",
    "Bamboo and Fields",
    "Fields  and Bamboo",
    "Fields or Secondary Forest/Pine Plantation",
]
subset = land_use[land_use["Classify"].isin(classes)]

# 4. Sum areas by class
area_by_class = (
    subset
    .groupby("Classify")["area_km2"]
    .sum()
    .reset_index()
    .rename(columns={"Classify": "class"})
)

# 5. Compute % of Jamaica (10 991 km²)
jamaica_total = 10991.0
area_by_class["percent_of_Jamaica"] = (
    area_by_class["area_km2"] / jamaica_total * 100
)

# 6. Calculate the combined total
total_percent = area_by_class["percent_of_Jamaica"].sum()
total_area    = area_by_class["area_km2"].sum()

# 7. Append a “Total” row via concat()
total_row = pd.DataFrame([{
    "class": "Total",
    "area_km2": total_area,
    "percent_of_Jamaica": total_percent
}])
area_by_class = pd.concat([area_by_class, total_row], ignore_index=True)

print(area_by_class)
print(f"Combined percent: {total_percent:.2f}%")

In [ ]:
land_use_hydrobasins_intersection = gpd.overlay(land_use, hydrobasins_clipped_saved, how='intersection')

In [ ]:
# Define the output shapefile path for the intersected data
output_intersection_shapefile = output_path / "land_use_hydrobasins_intersection.shp"

In [ ]:
# Save the intersected GeoDataFrame to a shapefile
land_use_hydrobasins_intersection.to_file(output_intersection_shapefile)

In [ ]:
# Now, read it back in to confirm it has been saved correctly
land_use_hydrobasins_intersection_saved = gpd.read_file(output_intersection_shapefile)

In [ ]:
# Print the first few rows to confirm the data is loaded
display(land_use_hydrobasins_intersection_saved.head())

In [ ]:
# Count the unique values in the 'HYBAS_ID' column to check how many rows I have
unique_hybas_ids = land_use_hydrobasins_intersection_saved['HYBAS_ID'].nunique()

# Print the result
print(f"Number of unique HYBAS_ID values: {unique_hybas_ids}")

### Step 3: Calculate total area of each catchment and the percentage catchment coverage of each land use type

#### 3.1 Calculate total area of each catchment

In [ ]:
# Calculate the area for each feature in the GeoDataFrame (in square units, based on CRS). 
# The units of the area will depend on the CRS of the data (EPSG: 3448, which uses meters, so the area is in square meters).
land_use_hydrobasins_intersection_saved['area'] = land_use_hydrobasins_intersection_saved.geometry.area

# Total area of each catchmemt - group by 'HYBAS_ID' and sum the 'area' for each catchment
total_area_by_catchment = land_use_hydrobasins_intersection_saved.groupby('HYBAS_ID')['area'].sum().reset_index()
total_catchment_area  = total_area_by_catchment.rename(columns={'area': 'total_catchment_area'})

display(total_catchment_area.head())


# after you’ve computed total_catchment_area as above:

# 1) Save to CSV (no index column)
total_catchment_area.to_csv('hydrobasin_catchment_areas.csv', index=False)

# 2) (Optional) Confirm it wrote correctly
print("Wrote", len(total_catchment_area), "rows to hydrobasin_catchment_areas.csv")

In [ ]:
# assuming you already have this:
# total_area_by_catchment = land_use_hydrobasins_intersection_saved.groupby('HYBAS_ID')['area'].sum().reset_index()
# total_catchment_area  = total_area_by_catchment.rename(columns={'area': 'total_catchment_area'})

# compute min, max and mean
min_area   = total_catchment_area['total_catchment_area'].min()
max_area   = total_catchment_area['total_catchment_area'].max()
mean_area  = total_catchment_area['total_catchment_area'].mean()

print(f"Smallest catchment: {min_area:,.0f} m²")
print(f"Largest  catchment: {max_area:,.0f} m²")
print(f"Average  catchment: {mean_area:,.0f} m²")

#### 3.2 Calculate area of each land use within each catchment

In [ ]:
# Area of land use within each catchmemt - group by 'HYBAS_ID' and 'classify' and sum the 'area' for each catchment
total_landuse_area_by_catchment = land_use_hydrobasins_intersection_saved.groupby(['HYBAS_ID', 'Classify'])['area'].sum().reset_index()

display(total_landuse_area_by_catchment.head())

# Define the output file path
output_landuse_areas_path = output_path / "total_landuse_area_by_catchment.csv"

# Export the DataFrame to a CSV file
total_landuse_area_by_catchment.to_csv(output_landuse_areas_path, index=False)

print(f"Data successfully exported to {output_landuse_areas_path}")

#### 3.3 Calculate percentage catchment covered by each land use type

In [ ]:
# Merge total area of catchments with the land use area to calculate percentage
land_use_with_total_area = total_landuse_area_by_catchment.merge(total_catchment_area, on='HYBAS_ID')

# Calculate the percentage of each land use area relative to the total catchment area
land_use_with_total_area['Percentage of Catchment'] = (land_use_with_total_area['area'] / land_use_with_total_area['total_catchment_area']) * 100

display(land_use_with_total_area.head())

# Define the output file path
output_percentage_landcover_csv_path = output_path / "land_use_percentage_by_catchment.csv"

# Export the DataFrame to a CSV file
land_use_with_total_area.to_csv(output_percentage_landcover_csv_path, index=False)

print(f"Data successfully exported to {output_percentage_landcover_csv_path}")

### Step 4: determine current forest coverage within each catchment and future afforestable area

#### 4.1 Determining current forest, non-afforestable and future afforestable land use categories

#### 4.2 Calculate the area of each land use category

In [ ]:
# ----------------------------------------------------------------------------
# Apply the function to each row and "explode" the dictionary so each row has a single key-value pair.
land_use_hydrobasins_intersection_saved['frac_dict'] = land_use_hydrobasins_intersection_saved.apply(Robyn_catchment_analysis.calculate_fractional_areas, axis=1)

expanded_rows = []
for idx, row in land_use_hydrobasins_intersection_saved.iterrows():
    for key, value in row['frac_dict'].items():
        new_row = row.copy()
        new_row['LandUseCategory'] = key
        new_row['Area'] = value
        expanded_rows.append(new_row)
        
expanded_gdf = gpd.GeoDataFrame(expanded_rows, crs=land_use_hydrobasins_intersection_saved.crs)
# ----------------------------------------------------------------------------


In [ ]:
# Calculate the forest flood equivalent area (i.e. the area that contributes to flood reduction)
forest_flood_equivalent_gdf = expanded_gdf[expanded_gdf['LandUseCategory'] == 'forest_flood_equivalent_classes']
forest_flood_equivalent_area = (
    forest_flood_equivalent_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index()
    .rename(columns={'Area': 'forest_flood_equivalent_area'})
)

# Calculate the afforestable area (including the agricultural component)
afforestable_agri_gdf = expanded_gdf[expanded_gdf['LandUseCategory'] == 'afforestable_including_agriculture']
afforestable_agri_area = (
    afforestable_agri_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index()
    .rename(columns={'Area': 'afforestable_including_agriculture_area'})
)

# Merge the results with the total catchment area DataFrame
final_summary = total_catchment_area.merge(
    forest_flood_equivalent_area, on='HYBAS_ID', how='left'
).merge(
    afforestable_agri_area, on='HYBAS_ID', how='left'
)

# Replace NaN values with 0 if some HYBAS_IDs don't have one of the categories
final_summary['forest_flood_equivalent_area'] = final_summary['forest_flood_equivalent_area'].fillna(0)
final_summary['afforestable_including_agriculture_area'] = final_summary['afforestable_including_agriculture_area'].fillna(0)

# Create a new column for the total future forest area including agriculture
final_summary['total_future_forest_area_including_agri'] = (
    final_summary['forest_flood_equivalent_area'] + final_summary['afforestable_including_agriculture_area']
)

# Calculate percentages relative to the total catchment area
final_summary['forest_flood_equivalent_percentage'] = (
    final_summary['forest_flood_equivalent_area'] / final_summary['total_catchment_area'] * 100
).round(2)
final_summary['afforestable_including_agriculture_percentage'] = (
    final_summary['afforestable_including_agriculture_area'] / final_summary['total_catchment_area'] * 100
).round(2)
final_summary['total_future_forest_including_agri_percentage'] = (
    final_summary['total_future_forest_area_including_agri'] / final_summary['total_catchment_area'] * 100
).round(2)

# Reorder columns for clarity
column_order = [
    'HYBAS_ID', 
    'total_catchment_area',
    'forest_flood_equivalent_area',
    'forest_flood_equivalent_percentage',
    'afforestable_including_agriculture_area',
    'afforestable_including_agriculture_percentage',
    'total_future_forest_area_including_agri',
    'total_future_forest_including_agri_percentage'
]
final_summary = final_summary[column_order]

# Display the final summary DataFrame
display(final_summary)

# Optional: Export the summary to CSV
output_forests = output_path / "catchment_forest_summary_with_percentages.csv"
final_summary.to_csv(output_forests, index=False)
print(f"Data successfully exported to {output_path}")

In [ ]:
# Step 3: Function to plot a specific land use category (updated for new category names)
def plot_land_use_category(category_name):
    """
    Plot the areas corresponding to a specific land use category.
    """
    # Filter by category
    filtered_data = expanded_gdf[expanded_gdf['LandUseCategory'] == category_name]
    
    if filtered_data.empty:
        print(f"No data found for category '{category_name}'")
        return
    
    # Plotting
    fig, ax = plt.subplots(1, 1, figsize=(12, 12))
    filtered_data.plot(
        column='Area',  # Optionally, color based on area
        cmap='Greens',  # Adjust colormap as needed
        legend=True,
        ax=ax
    )

    # Plot the boundary (assumes jamaica_boundary is a GeoDataFrame with a 'boundary')
    jamaica_boundary.boundary.plot(
        ax=ax, color='black', linewidth=1.5, label='Jamaica Boundary'
    )
    
    # Title and labels
    plt.title(f"Land Use Category: {category_name.replace('_', ' ').title()}", fontsize=16)
    plt.xlabel("Longitude", fontsize=12)
    plt.ylabel("Latitude", fontsize=12)
    
    # Adjust legend position if present
    legend = ax.get_legend()
    if legend:
        legend.set_bbox_to_anchor((1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()

# Example usage with the updated category name:
plot_land_use_category('afforestable_including_agriculture')

In [ ]:
# Step 1: Aggregate Area by Catchment and Land Use Category
category_area = expanded_gdf.groupby(['HYBAS_ID', 'LandUseCategory'])['Area'].sum().reset_index()

# Step 2: Calculate Total Area per Catchment from expanded_gdf
total_area = expanded_gdf.groupby('HYBAS_ID')['Area'].sum().reset_index().rename(columns={'Area': 'TotalArea'})

# Step 3: Merge Aggregated Data with Total Area
category_percentage = pd.merge(category_area, total_area, on='HYBAS_ID')

# Step 4: Compute Percentage per Land Use Category
category_percentage['Percentage'] = (category_percentage['Area'] / category_percentage['TotalArea']) * 100

# (Optional) Step 5: Pivot Data for Easier Interpretation
percentage_pivot = category_percentage.pivot(index='HYBAS_ID', columns='LandUseCategory', values='Percentage').fillna(0).reset_index()

# Display the percentage DataFrame
display("Percentage of Each Land Use Category within Each Catchment:")
display(percentage_pivot.head())

In [ ]:
# Display the first few rows to verify
display(percentage_pivot.head())

In [ ]:
display("Aggregated summary by HYBAS_ID:")
display(final_summary.head())

In [ ]:
# Define the output path and export percentage pivot DataFrame
output_hydrobasins_percentage_afforestable_csv_path = output_path / "afforestable_percentages_by_catchment.csv"
percentage_pivot.to_csv(output_hydrobasins_percentage_afforestable_csv_path, index=False)

In [ ]:
# 1. Pivot the area data by HYBAS_ID and LandUseCategory
area_pivot = category_percentage.pivot(
    index='HYBAS_ID',
    columns='LandUseCategory',
    values='Area'
).fillna(0).reset_index()

# 2. Merge the area data into percentage_pivot
percentage_pivot = pd.merge(percentage_pivot, area_pivot, on='HYBAS_ID', suffixes=('_pct', '_area'))

# 3. Add the total area for each hydrobasin.
# This assumes that the area columns from the pivot start at a fixed position; adjust if needed.
percentage_pivot['TotalArea'] = percentage_pivot.iloc[:, len(percentage_pivot.columns) - len(area_pivot.columns) + 1:].sum(axis=1)

# Define the export path for percentages including catchment area and export the DataFrame
output_hydrobasins_percentage_afforestable_with_catchment_area_csv_path = output_path / "afforestable_percentages_by_catchment_including_area.csv"
percentage_pivot.to_csv(output_hydrobasins_percentage_afforestable_with_catchment_area_csv_path, index=False)

# Add a new column for TotalArea in km² by dividing by 1,000,000
percentage_pivot['TotalArea_km2'] = percentage_pivot['TotalArea'] / 1_000_000

# Display the updated DataFrame to check the new column
display(percentage_pivot.head())

In [ ]:
# Export the final DataFrame with total afforestable area by catchment
total_afforestable_area_by_catchment_csv_path = output_path / "total_afforestable_area_by_catchment.csv"
percentage_pivot.to_csv(total_afforestable_area_by_catchment_csv_path, index=False)

In [ ]:

# 1) Read mapping of HYBAS_ID → new_id
mapping_df = pd.read_csv(
    Path(output_dir)/"catchment_connectivity.csv",
    dtype={"HYBAS_ID": str}
)[['HYBAS_ID','new_id']]

# 2) Build hydro_summary with both % cols
hydro_summary = hydrobasins_clipped_saved.merge(
    final_summary[[
        'HYBAS_ID',
        'forest_flood_equivalent_percentage',
        'total_future_forest_including_agri_percentage'
    ]],
    on='HYBAS_ID', how='left'
).fillna(0)

# 3) Cast HYBAS_ID to str & merge in new_id
hydro_summary['HYBAS_ID'] = hydro_summary['HYBAS_ID'].astype(str)
hydro_summary = hydro_summary.merge(mapping_df, on='HYBAS_ID', how='left')
assert hydro_summary['new_id'].notna().all(), "Missing some new_id!"

expanded_gdf['HYBAS_ID'] = expanded_gdf['HYBAS_ID'].astype(str)

# 4) Forest‐only & Future‐forest GeoDataFrames
forest_gdf = (
    expanded_gdf[expanded_gdf['LandUseCategory']=='forest_flood_equivalent_classes']
    .to_crs(jamaica_metric_grid_crs)
    .merge(hydro_summary[['HYBAS_ID','forest_flood_equivalent_percentage','new_id']],
           on='HYBAS_ID', how='left')
    .fillna(0)
)
future_gdf = (
    expanded_gdf[expanded_gdf['LandUseCategory']=='afforestable_including_agriculture']
    .to_crs(jamaica_metric_grid_crs)
    .merge(hydro_summary[['HYBAS_ID','total_future_forest_including_agri_percentage','new_id']],
           on='HYBAS_ID', how='left')
    .fillna(0)
)

# 5) Shared Greens ramp
cmap = plt.colormaps['Greens']
vmax = forest_gdf['forest_flood_equivalent_percentage'].max()
norm = mpl.colors.Normalize(vmin=0, vmax=vmax)
forest_gdf['color'] = forest_gdf['forest_flood_equivalent_percentage']\
    .map(lambda pct: mcolors.to_hex(cmap(norm(pct))))
future_gdf['color'] = future_gdf['total_future_forest_including_agri_percentage']\
    .map(lambda pct: mcolors.to_hex(cmap(norm(pct))))

# 6) Scale bar (bottom right) & north arrow (top right)
def add_scale_bar(ax, length_km=20, loc=(0.9,0.05)):
    x,y = loc; half=0.05
    ax.plot([x-half,x+half],[y,y],transform=ax.transAxes,color='black',lw=2)
    for p in (x-half,x,x+half):
        ax.plot([p,p],[y-0.005,y+0.005],transform=ax.transAxes,color='black',lw=2)
    ax.text(x-half,y-0.03,'0',transform=ax.transAxes,ha='center',va='center')
    ax.text(x,y-0.03,f'{length_km//2}',transform=ax.transAxes,ha='center',va='center')
    ax.text(x+half,y-0.03,f'{length_km}',transform=ax.transAxes,ha='center',va='center')
    ax.text(x+half+0.02,y,'km',transform=ax.transAxes,ha='left',va='center')

def add_north_arrow(ax, loc=(0.9,0.9), size=0.05):
    x,y = loc
    ax.annotate('', xy=(x,y+size), xycoords='axes fraction',
                xytext=(x,y), textcoords='axes fraction',
                arrowprops=dict(facecolor='black',edgecolor='black',
                                headwidth=10,headlength=15,width=5))
    ax.text(x,y+size+0.02,'N', transform=ax.transAxes,
            ha='center', va='center', fontsize=12, fontweight='bold')

# 7) Plot & save function
def plot_and_save(gdf, pct_col, pct_label, title, fname):
    fig, ax = plt.subplots(figsize=(18,14), dpi=300)
    # patches
    gdf.plot(ax=ax, color=gdf['color'], linewidth=0, alpha=0.8)
    # boundaries
    hydro_summary.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5)
    # colorbar
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm._A=[]
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label(f"{pct_label} (% of catchment)", family='Times New Roman')
    # new_id labels
    for _,r in hydro_summary.iterrows():
        pt = r.geometry.representative_point()
        ax.text(pt.x, pt.y, str(int(r.new_id)),
                ha='center', va='center', fontsize=6, weight='bold')
    # legend
    h = Line2D([0],[0], color='black', lw=0.5, label='Catchment boundary')
    ax.legend(handles=[h], title='Legend',
              bbox_to_anchor=(0.5,-0.1), loc='upper center',
              frameon=False, fontsize=12, title_fontsize=14,
              prop={'family':'Times New Roman'}
    ).get_title().set_position((0,10))
    # scale bar & north
    add_scale_bar(ax, loc=(0.9,0.05))
    add_north_arrow(ax, loc=(0.9,0.9))
    # finish
    ax.set_axis_off()
    plt.title(title, fontsize=20, fontweight='bold',
              fontname='Times New Roman', pad=20)
    plt.tight_layout()
    fig.savefig(Path(output_dir)/f"{fname}.png", dpi=300, bbox_inches='tight')
    fig.savefig(Path(output_dir)/f"{fname}.pdf",                bbox_inches='tight')
    plt.show()

# 8) Draw & save panels
plot_and_save(
    forest_gdf,
    'forest_flood_equivalent_percentage',
    'Baseline forest cover',
    "Figure 1(a) Baseline forest area shaded by catchment-level % cover",
    "Figure1a_baseline_forest"
)
plot_and_save(
    future_gdf,
    'total_future_forest_including_agri_percentage',
    'Total future forest cover',
    "Figure 1(b) Total future forest area shaded by catchment-level % cover",
    "Figure1b_future_forest"
)

In [ ]:

# 1) Read your mapping of HYBAS_ID → new_id (1–103)
mapping_df = pd.read_csv(
    Path(output_dir)/"catchment_connectivity.csv",
    dtype={"HYBAS_ID": str}
)[['HYBAS_ID','new_id']]

# 2) Rebuild hydro_summary with both % columns
hydro_summary = hydrobasins_clipped_saved.merge(
    final_summary[[
        'HYBAS_ID',
        'forest_flood_equivalent_percentage',
        'total_future_forest_including_agri_percentage'
    ]],
    on='HYBAS_ID', how='left'
).fillna(0)

# 3) Cast HYBAS_ID to str everywhere and merge in new_id
hydro_summary['HYBAS_ID'] = hydro_summary['HYBAS_ID'].astype(str)
hydro_summary = hydro_summary.merge(mapping_df, on='HYBAS_ID', how='left')
assert hydro_summary['new_id'].notna().all(), "Some catchments missing new_id!"

expanded_gdf['HYBAS_ID'] = expanded_gdf['HYBAS_ID'].astype(str)

# 4) Build the forest and future‐forest GeoDataFrames
forest_gdf = (
    expanded_gdf[expanded_gdf['LandUseCategory']=='forest_flood_equivalent_classes']
    .to_crs(jamaica_metric_grid_crs)
    .merge(
        hydro_summary[['HYBAS_ID','forest_flood_equivalent_percentage','new_id']],
        on='HYBAS_ID', how='left'
    )
    .fillna(0)
)

future_gdf = (
    expanded_gdf[
        expanded_gdf['LandUseCategory'].isin([
            'forest_flood_equivalent_classes',
            'afforestable_including_agriculture'
        ])
    ]
    .to_crs(jamaica_metric_grid_crs)
    .merge(
        hydro_summary[['HYBAS_ID','total_future_forest_including_agri_percentage','new_id']],
        on='HYBAS_ID', how='left'
    )
    .fillna(0)
)

# 5) Shared “Greens” ramp normalized to baseline max
cmap = plt.colormaps['Greens']
vmax = forest_gdf['forest_flood_equivalent_percentage'].max()
norm = mpl.colors.Normalize(vmin=0, vmax=vmax)

forest_gdf['color'] = forest_gdf['forest_flood_equivalent_percentage']\
    .map(lambda pct: mcolors.to_hex(cmap(norm(pct))))
future_gdf['color'] = future_gdf['total_future_forest_including_agri_percentage']\
    .map(lambda pct: mcolors.to_hex(cmap(norm(pct))))

# 6) Scale bar & north arrow
def add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, tick_height=0.01, label_offset=0.04, km_offset=0.01):
    half = 0.05; x,y = location
    ax.plot([x-half,x+half],[y,y],transform=ax.transAxes,color='black',lw=2)
    for pos in (x-half,x,x+half):
        ax.plot([pos,pos],[y-0.005,y+0.005],transform=ax.transAxes,color='black',lw=2)
    ax.text(x-half,y-0.03,"0",transform=ax.transAxes,ha='center',va='center')
    ax.text(x,y-0.03,f"{length_km//2}",transform=ax.transAxes,ha='center',va='center')
    ax.text(x+half,y-0.03,f"{length_km}",transform=ax.transAxes,ha='center',va='center')
    ax.text(x+half+0.02,y,"km",transform=ax.transAxes,ha='left',va='center')

def add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03):
    x,y = location
    ax.annotate(
        "", 
        xy=(x, y+size), xycoords='axes fraction',
        xytext=(x, y), textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black',
                        headwidth=10, headlength=15, width=5)
    )
    ax.text(
        x, y+size+0.02, "N",
        transform=ax.transAxes, ha='center', va='center',
        fontsize=12, fontweight='bold'
    )

# 7) Single function to both plot and save
def plot_and_save(gdf, pct_label, title, fname):
    fig, ax = plt.subplots(figsize=(18,14), dpi=300)

    # forest patches
    gdf.plot(ax=ax, color=gdf['color'], linewidth=0, alpha=0.8)

    # catchment boundaries
    hydro_summary.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5)

    # continuous colorbar
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm._A=[]
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label(f"{pct_label} (% of catchment)", family='Times New Roman')

    # catchment IDs
    for _, row in hydro_summary.iterrows():
        pt = row.geometry.representative_point()
        ax.text(pt.x, pt.y, str(int(row.new_id)),
                ha='center', va='center', fontsize=6, fontweight='bold')

    # legend (just the boundary)
    handle = Line2D([0],[0], color='black', lw=0.5, label='Catchment boundary')
    ax.legend(handles=[handle], title='Legend',
              bbox_to_anchor=(0.5, -0.1), loc='upper center',
              frameon=False, fontsize=12, title_fontsize=14,
              prop={'family':'Times New Roman'}).get_title().set_position((0,10))

    # both up top
    add_scale_bar(ax)
    add_north_arrow(ax)

    ax.set_axis_off()
    plt.title(title, fontsize=20, fontweight='bold',
              fontname='Times New Roman', pad=20)
    plt.tight_layout()

    # save both formats
    out_png = Path(output_dir)/f"{fname}.png"
    out_pdf = Path(output_dir)/f"{fname}.pdf"
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    fig.savefig(out_pdf,              bbox_inches='tight')
    print(f"Saved {out_png} and {out_pdf}")

    plt.show()

# 8) Finally, call it for both panels:
plot_and_save(
    forest_gdf,
    pct_label="Baseline forest cover",
    title="Figure 1(a) Baseline forest area shaded by catchment-level % cover",
    fname="Figure1a_baseline_forest"
)

plot_and_save(
    future_gdf,
    pct_label="Total future forest cover",
    title="Figure 1(b) Total future forest area shaded by catchment-level % cover",
    fname="Figure1b_future_forest"
)

In [ ]:

# # 1) Read your mapping of HYBAS_ID → new_id (1–103)
# mapping_df = pd.read_csv(
#     Path(output_dir)/"catchment_connectivity.csv",
#     dtype={"HYBAS_ID": str}
# )[['HYBAS_ID','new_id']]

# # 2) Rebuild hydro_summary with both % columns
# hydro_summary = hydrobasins_clipped_saved.merge(
#     final_summary[[
#         'HYBAS_ID',
#         'forest_flood_equivalent_percentage',
#         'total_future_forest_including_agri_percentage'
#     ]],
#     on='HYBAS_ID', how='left'
# ).fillna(0)

# # 3) Cast HYBAS_ID to str everywhere and merge in new_id
# hydro_summary['HYBAS_ID'] = hydro_summary['HYBAS_ID'].astype(str)
# hydro_summary = hydro_summary.merge(mapping_df, on='HYBAS_ID', how='left')
# assert hydro_summary['new_id'].notna().all(), "Some catchments missing new_id!"

# expanded_gdf['HYBAS_ID'] = expanded_gdf['HYBAS_ID'].astype(str)

# # 4) Build the forest and future‐forest GeoDataFrames
# forest_gdf = (
#     expanded_gdf[expanded_gdf['LandUseCategory']=='forest_flood_equivalent_classes']
#     .to_crs(jamaica_metric_grid_crs)
#     .merge(
#         hydro_summary[['HYBAS_ID','forest_flood_equivalent_percentage','new_id']],
#         on='HYBAS_ID', how='left'
#     )
#     .fillna(0)
# )

# future_gdf = (
#     expanded_gdf[
#         expanded_gdf['LandUseCategory'].isin([
#             'forest_flood_equivalent_classes',
#             'afforestable_including_agriculture'
#         ])
#     ]
#     .to_crs(jamaica_metric_grid_crs)
#     .merge(
#         hydro_summary[['HYBAS_ID','total_future_forest_including_agri_percentage','new_id']],
#         on='HYBAS_ID', how='left'
#     )
#     .fillna(0)
# )

# # 5) Shared “Greens” ramp normalized to baseline max
# cmap = plt.colormaps['Greens']
# vmax = forest_gdf['forest_flood_equivalent_percentage'].max()
# norm = mpl.colors.Normalize(vmin=0, vmax=vmax)

# forest_gdf['color'] = forest_gdf['forest_flood_equivalent_percentage']\
#     .map(lambda pct: mcolors.to_hex(cmap(norm(pct))))
# future_gdf['color'] = future_gdf['total_future_forest_including_agri_percentage']\
#     .map(lambda pct: mcolors.to_hex(cmap(norm(pct))))

# # # 6) Scale bar & north arrow
# # def add_scale_bar(ax, length_km=20, location=(0.9,0.08)):
# #     half = 0.05
# #     x,y = location
# #     ax.plot([x-half,x+half],[y,y],transform=ax.transAxes,color='black',lw=2)
# #     for pos in (x-half,x,x+half):
# #         ax.plot([pos,pos],[y-0.005,y+0.005],transform=ax.transAxes,color='black',lw=2)
# #     ax.text(x-half,y-0.03,"0",transform=ax.transAxes,ha='center',va='center')
# #     ax.text(x,y-0.03,f"{length_km//2}",transform=ax.transAxes,ha='center',va='center')
# #     ax.text(x+half,y-0.03,f"{length_km}",transform=ax.transAxes,ha='center',va='center')
# #     ax.text(x+half+0.02,y,"km",transform=ax.transAxes,ha='left',va='center')

# # def add_north_arrow(ax, location=(0.9,0.85), size=0.05):
# #     x,y = location
# #     ax.annotate("",xy=(x,y+size),xycoords='axes fraction',
# #                 xytext=(x,y),textcoords='axes fraction',
# #                 arrowprops=dict(facecolor='black',edgecolor='black',
# #                                 headwidth=10,headlength=15,width=5))
# #     ax.text(x,y+size+0.02,"N",transform=ax.transAxes,ha='center',va='center',
# #             fontsize=12,fontweight='bold')

# # # 7) Plot & save function
# # def plot_and_save(gdf, pct_label, title, fname):
# #     fig, ax = plt.subplots(figsize=(18,14),dpi=300)

# #     # forest patches
# #     gdf.plot(ax=ax, color=gdf['color'], linewidth=0, alpha=0.8)

# #     # boundaries
# #     hydro_summary.boundary.plot(ax=ax,edgecolor='black',linewidth=0.5)

# #     # colorbar
# #     sm = mpl.cm.ScalarMappable(cmap=cmap,norm=norm); sm._A=[]
# #     cbar=fig.colorbar(sm,ax=ax,fraction=0.03,pad=0.04)
# #     cbar.set_label(f"{pct_label} (% of catchment)",family='Times New Roman')

# #     # new_id labels
# #     for _,r in hydro_summary.iterrows():
# #         pt=r.geometry.representative_point()
# #         ax.text(pt.x,pt.y,str(int(r.new_id)),
# #                 ha='center',va='center',fontsize=6,weight='bold')

# #     # legend (just boundary entry)
# #     handle = Line2D([0],[0],color='black',lw=0.5,label='Catchment boundary')
# #     ax.legend(handles=[handle],title='Legend',
# #               bbox_to_anchor=(0.5,-0.1),loc='upper center',
# #               frameon=False,fontsize=12,title_fontsize=14,
# #               prop={'family':'Times New Roman'}).get_title().set_position((0,10))

# #     # scale & north
# #     add_scale_bar(ax); add_north_arrow(ax)

# #     # title & finish
# #     ax.set_axis_off()
# #     plt.title(title,fontsize=20,fontweight='bold',
# #               fontname='Times New Roman',pad=20)
# #     plt.tight_layout()

# #     # save PNG + PDF
# #     fig.savefig(Path(output_dir)/f"{fname}.png",dpi=300,bbox_inches='tight')
# #     fig.savefig(Path(output_dir)/f"{fname}.pdf",           bbox_inches='tight')

# #     plt.show()

# # # 8) Draw panels:
# # plot_and_save(
# #     forest_gdf,
# #     pct_label="Baseline forest cover",
# #     title="Figure 1(a) Baseline forest area shaded by catchment-level % cover",
# #     fname="Figure1a_baseline_forest"
# # )

# # plot_and_save(
# #     future_gdf,
# #     pct_label="Total future forest cover",
# #     title="Figure 1(b) Total future forest area shaded by catchment-level % cover",
# #     fname="Figure1b_future_forest"
# # )


# # 6) Scale bar & north arrow
# def add_scale_bar(ax, length_km=20, location=(0.9,0.08)):
#     half = 0.05
#     x,y = location
#     ax.plot([x-half,x+half],[y,y],transform=ax.transAxes,color='black',lw=2)
#     for pos in (x-half,x,x+half):
#         ax.plot([pos,pos],[y-0.005,y+0.005],transform=ax.transAxes,color='black',lw=2)
#     ax.text(x-half,y-0.03,"0",transform=ax.transAxes,ha='center',va='center')
#     ax.text(x,y-0.03,f"{length_km//2}",transform=ax.transAxes,ha='center',va='center')
#     ax.text(x+half,y-0.03,f"{length_km}",transform=ax.transAxes,ha='center',va='center')
#     ax.text(x+half+0.02,y,"km",transform=ax.transAxes,ha='left',va='center')

# def add_north_arrow(ax, location=(0.95,0.95), size=0.05):
#     x,y = location
#     ax.annotate(
#         "", 
#         xy=(x, y+size), xycoords='axes fraction',
#         xytext=(x, y), textcoords='axes fraction',
#         arrowprops=dict(facecolor='black', edgecolor='black',
#                         headwidth=10, headlength=15, width=5)
#     )
#     ax.text(
#         x, y+size+0.02, "N",
#         transform=ax.transAxes, ha='center', va='center',
#         fontsize=12, fontweight='bold'
#     )

# # 7) Plot & save function
# def plot_and_save(gdf, pct_label, title, fname):
#     fig, ax = plt.subplots(figsize=(18,14), dpi=300)

#     # forest patches
#     gdf.plot(ax=ax, color=gdf['color'], linewidth=0, alpha=0.8)

#     # boundaries
#     hydro_summary.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5)

#     # colorbar
#     sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm._A=[]
#     cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
#     cbar.set_label(f"{pct_label} (% of catchment)", family='Times New Roman')

#     # new_id labels
#     for _, r in hydro_summary.iterrows():
#         pt = r.geometry.representative_point()
#         ax.text(
#             pt.x, pt.y, str(int(r.new_id)),
#             ha='center', va='center', fontsize=6, weight='bold'
#         )

#     # legend (just boundary entry)
#     handle = Line2D([0],[0], color='black', lw=0.5, label='Catchment boundary')
#     ax.legend(
#         handles=[handle],
#         title='Legend',
#         bbox_to_anchor=(0.5, -0.1),
#         loc='upper center',
#         frameon=False,
#         fontsize=12,
#         title_fontsize=14,
#         prop={'family':'Times New Roman'}
#     ).get_title().set_position((0,10))

#     # scale bar & north arrow (north arrow now sits at top right)
#     add_scale_bar(ax)
#     add_north_arrow(ax)

#     # title & finish
#     ax.set_axis_off()
#     plt.title(
#         title,
#         fontsize=20, fontweight='bold',
#         fontname='Times New Roman', pad=20
#     )
#     plt.tight_layout()

#     # save PNG + PDF
#     fig.savefig(Path(output_dir)/f"{fname}.png", dpi=300, bbox_inches='tight')
#     fig.savefig(Path(output_dir)/f"{fname}.pdf",                    bbox_inches='tight')
#     plt.show()

In [ ]:
# # # 1) Read and merge your custom new_id mapping
# # mapping_path = Path(output_dir) / "catchment_connectivity.csv"
# # mapping_df = pd.read_csv(mapping_path, dtype={"HYBAS_ID": str})[['HYBAS_ID','new_id']]

# # # 2) Rebuild hydro_summary so it includes both % columns
# # hydro_summary = hydrobasins_clipped_saved.merge(
# #     final_summary[[
# #         'HYBAS_ID',
# #         'forest_flood_equivalent_percentage',
# #         'total_future_forest_including_agri_percentage'
# #     ]],
# #     on='HYBAS_ID',
# #     how='left'
# # ).fillna({
# #     'forest_flood_equivalent_percentage': 0,
# #     'total_future_forest_including_agri_percentage': 0
# # })


# # # ─── After rebuilding hydro_summary ────────────────────────────────────────────
# # # Ensure expanded_gdf has HYBAS_ID as str as well
# # expanded_gdf['HYBAS_ID'] = expanded_gdf['HYBAS_ID'].astype(str)


# # # 3) Merge in new_id and validate
# # hydro_summary['HYBAS_ID'] = hydro_summary['HYBAS_ID'].astype(str)
# # hydro_summary = hydro_summary.merge(mapping_df, on='HYBAS_ID', how='left')
# # missing = hydro_summary['new_id'].isna().sum()
# # if missing:
# #     raise ValueError(f"{missing} catchments failed to match on HYBAS_ID")

# # # 4) Build forest_gdf and future_forest_gdf with their % and new_id
# # forest_gdf = (
# #     expanded_gdf[expanded_gdf['LandUseCategory']=='forest_flood_equivalent_classes']
# #     .to_crs(jamaica_metric_grid_crs)
# #     .merge(
# #         hydro_summary[['HYBAS_ID','forest_flood_equivalent_percentage','new_id']],
# #         on='HYBAS_ID', how='left'
# #     )
# #     .fillna({'forest_flood_equivalent_percentage': 0})
# # )

# # future_forest_gdf = (
# #     expanded_gdf[
# #         expanded_gdf['LandUseCategory'].isin([
# #             'forest_flood_equivalent_classes',
# #             'afforestable_including_agriculture'
# #         ])
# #     ]
# #     .to_crs(jamaica_metric_grid_crs)
# #     .merge(
# #         hydro_summary[['HYBAS_ID','total_future_forest_including_agri_percentage','new_id']],
# #         on='HYBAS_ID', how='left'
# #     )
# #     .fillna({'total_future_forest_including_agri_percentage': 0})
# # )

# # # 5) Create a shared “Greens” ramp normalized to the baseline max %
# # cmap = plt.colormaps['Greens']
# # vmax_shared = forest_gdf['forest_flood_equivalent_percentage'].max()
# # norm = mpl.colors.Normalize(vmin=0, vmax=vmax_shared)

# # forest_gdf['color'] = forest_gdf['forest_flood_equivalent_percentage']\
# #     .apply(lambda pct: mcolors.to_hex(cmap(norm(pct))))
# # future_forest_gdf['color'] = future_forest_gdf['total_future_forest_including_agri_percentage']\
# #     .apply(lambda pct: mcolors.to_hex(cmap(norm(pct))))

# # # 6) Define scale bar & north arrow helpers
# # def add_scale_bar(ax, length_km=20, location=(0.9, 0.08),
# #                   linewidth=2, tick_height=0.01,
# #                   label_offset=0.02, km_offset=0.03):
# #     x, y = location
# #     half = 0.05
# #     ax.plot([x-half, x+half], [y, y], transform=ax.transAxes,
# #             color='black', linewidth=linewidth)
# #     for pos in (x-half, x, x+half):
# #         ax.plot([pos, pos], [y-tick_height/2, y+tick_height/2],
# #                 transform=ax.transAxes, color='black', linewidth=linewidth)
# #     ax.text(x-half, y-tick_height-label_offset, "0",
# #             transform=ax.transAxes, ha='center', va='center', fontsize=10)
# #     ax.text(x, y-tick_height-label_offset, f"{length_km//2}",
# #             transform=ax.transAxes, ha='center', va='center', fontsize=10)
# #     ax.text(x+half, y-tick_height-label_offset, f"{length_km}",
# #             transform=ax.transAxes, ha='center', va='center', fontsize=10)
# #     ax.text(x+half+km_offset, y, "km",
# #             transform=ax.transAxes, ha='left', va='center', fontsize=12)

# # def add_north_arrow(ax, location=(0.9, 0.15), size=0.05,
# #                     fontsize=12, label_offset=0.03):
# #     x, y = location
# #     ax.annotate(
# #         '', xy=(x, y+size), xycoords='axes fraction',
# #         xytext=(x, y), textcoords='axes fraction',
# #         arrowprops=dict(facecolor='black', edgecolor='black',
# #                         headwidth=10, headlength=15, width=5)
# #     )
# #     ax.text(x, y+size+label_offset, "N", transform=ax.transAxes,
# #             fontsize=fontsize, fontweight="bold",
# #             ha="center", va="center", color="black")

# # # 7) Plotting function
# # def plot_with_labels(gdf, cmap, norm, title, pct_col):
# #     fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# #     # forest patches
# #     gdf.plot(ax=ax, color=gdf['color'], linewidth=0, alpha=0.8)

# #     # catchment boundaries
# #     hydro_summary.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5)

# #     # colorbar
# #     sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm._A = []
# #     cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
# #     cbar.set_label(f'{title.split()[1]} cover (% of catchment)', fontsize=12,
# #                    family='Times New Roman')

# #     # new_id labels
# #     for _, row in hydro_summary.iterrows():
# #         pt = row.geometry.representative_point()
# #         ax.text(
# #             pt.x, pt.y, str(int(row.new_id)),
# #             fontsize=6, ha='center', va='center',
# #             color='black', fontweight='bold'
# #         )

# #     # legend (boundaries only)
# #     handle = Line2D([0], [0], color='black', linewidth=0.5,
# #                     label='Catchment boundary')
# #     legend = ax.legend(
# #         handles=[handle],
# #         title='Legend',
# #         bbox_to_anchor=(0.5, -0.1),
# #         loc='upper center',
# #         frameon=False,
# #         fontsize=12,
# #         title_fontsize=14,
# #         prop={'family': 'Times New Roman'}
# #     )
# #     legend.get_title().set_position((0, 10))

# #     # scale bar & north arrow
# #     add_scale_bar(ax)
# #     add_north_arrow(ax)

# #     plt.title(title, fontsize=20, fontweight='bold',
# #               fontname='Times New Roman', loc='center', pad=20)
# #     ax.set_axis_off()
# #     plt.tight_layout()
# #     plt.show()

# # # 8) Finally, draw your two panels:
# # plot_with_labels(
# #     forest_gdf, cmap, norm,
# #     "Figure 1(a) Baseline Forest patches", 'forest_flood_equivalent_percentage'
# # )
# # plot_with_labels(
# #     future_forest_gdf, cmap, norm,
# #     "Figure 1(b) Total Future Forest patches", 'total_future_forest_including_agri_percentage'
# # )


# # # right after plotting but before plt.show():

# # # for the baseline map
# # fig.savefig(
# #     output_path / "Figure1a_baseline_forest.png", 
# #     dpi=300,                   # 300 dpi is publication-standard for raster
# #     bbox_inches='tight'
# # )
# # fig.savefig(
# #     output_path / "Figure1a_baseline_forest.pdf", 
# #     bbox_inches='tight'       # PDF will scale perfectly
# # )

# # # for the future-forest map
# # fig2.savefig(
# #     output_dir / "Figure1b_future_forest.png", 
# #     dpi=300, 
# #     bbox_inches='tight'
# # )
# # fig2.savefig(
# #     output_dir / "Figure1b_future_forest.pdf", 
# #     bbox_inches='tight'
# # )

# # Figure 1(a)
# fig1, _ = plot_with_labels(
#     forest_gdf, cmap, norm,
#     "Figure 1(a) Baseline Forest patches",
#     'forest_flood_equivalent_percentage',
#     filename="Figure1a_baseline_forest.png"
# )
# # Also save a PDF version:
# fig1.savefig(output_dir / "Figure1a_baseline_forest.pdf", bbox_inches='tight')

# # Figure 1(b)
# fig2, _ = plot_with_labels(
#     future_forest_gdf, cmap, norm,
#     "Figure 1(b) Total Future Forest patches",
#     'total_future_forest_including_agri_percentage',
#     filename="Figure1b_future_forest.png"
# )
# fig2.savefig(output_dir / "Figure1b_future_forest.pdf", bbox_inches='tight')

In [ ]:
# 1) Paths
output_path  = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data")
mapping_path = Path(output_dir) / "catchment_connectivity.csv"

# 2) Read in the two tables
final_summary = pd.read_csv(
    output_path / "catchment_forest_summary_with_percentages.csv",
    dtype={"HYBAS_ID": str}     # read as string
)
mapping_df = (
    pd.read_csv(mapping_path, dtype={"HYBAS_ID": str})
      [['HYBAS_ID','new_id']]
)

# 3) Ensure final_summary.HYBAS_ID is string, too
final_summary['HYBAS_ID'] = final_summary['HYBAS_ID'].astype(str)

# 4) Merge them
table = final_summary.merge(mapping_df, on="HYBAS_ID", how="left")

# 5) Convert areas from m² to km²
table["Baseline forest area (km²)"]  = table["forest_flood_equivalent_area"]            / 1e6
table["Afforestable area (km²)"]     = table["afforestable_including_agriculture_area"] / 1e6


# # 6a) Select & rename columns
# summary_table = table[[
#     "new_id",
#     "Baseline forest area (km²)",
#     "forest_flood_equivalent_percentage",
#     "Afforestable area (km²)",
#     "afforestable_including_agriculture_percentage"
# ]].rename(columns={
#     "new_id":                               "Catchment ID",
#     "forest_flood_equivalent_percentage":   "Baseline forest (%)",
#     "Afforestable area (km²)": "Total future forest (km²)",
#     "afforestable_including_agriculture_percentage": "Total future forest (%)"
# })

# # 6b) Round the numeric columns to 2 decimal places
# summary_table[[
#     "Baseline forest area (km²)",
#     "Baseline forest (%)",
#     "Total future forest (km²)",
#     "Total future forest (%)"
# ]] = summary_table[[
#     "Baseline forest area (km²)",
#     "Baseline forest (%)",
#     "Total future forest (km²)",
#     "Total future forest (%)"
# ]].round(2)

# # 6c) Add difference columns
# summary_table['Area difference (km²)'] = (
#     summary_table['Total future forest (km²)']
#     - summary_table['Baseline forest area (km²)']
# )

# summary_table['Percentage difference (%)'] = (
#     summary_table['Total future forest (%)']
#     - summary_table['Baseline forest (%)']
# )

# # Round those new columns too
# summary_table[['Area difference (km²)', 'Percentage difference (%)']] = (
#     summary_table[['Area difference (km²)', 'Percentage difference (%)']]
#     .round(2)
# )


# 6a) Select & rename the correct columns
summary_table = table[[
    "new_id",
    "forest_flood_equivalent_area",
    "forest_flood_equivalent_percentage",
    "total_future_forest_area_including_agri",
    "total_future_forest_including_agri_percentage"
]].rename(columns={
    "new_id":                                 "Catchment ID",
    "forest_flood_equivalent_area":           "Baseline forest area (km²)",
    "forest_flood_equivalent_percentage":     "Baseline forest (%)",
    "total_future_forest_area_including_agri":"Total future forest (km²)",
    "total_future_forest_including_agri_percentage": "Total future forest (%)"
})

# 6b) Convert m²→km² for the two area columns
summary_table["Baseline forest area (km²)"]    = (
    summary_table["Baseline forest area (km²)"] / 1e6
)
summary_table["Total future forest (km²)"]     = (
    summary_table["Total future forest (km²)"] / 1e6
)

# 6c) Round all four numeric columns to 2 d.p.
summary_table[[
    "Baseline forest area (km²)",
    "Baseline forest (%)",
    "Total future forest (km²)",
    "Total future forest (%)"
]] = summary_table[[
    "Baseline forest area (km²)",
    "Baseline forest (%)",
    "Total future forest (km²)",
    "Total future forest (%)"
]].round(2)

# 6d) Add difference columns and round them
summary_table["Area difference (km²)"] = (
    summary_table["Total future forest (km²)"]
    - summary_table["Baseline forest area (km²)"]
)
summary_table["Percentage difference (%)"] = (
    summary_table["Total future forest (%)"]
    - summary_table["Baseline forest (%)"]
)
summary_table[[
    "Area difference (km²)",
    "Percentage difference (%)"
]] = summary_table[[
    "Area difference (km²)",
    "Percentage difference (%)"
]].round(2)



# 7) Sort by Catchment ID
summary_table = summary_table.sort_values("Catchment ID").reset_index(drop=True)

# 8) Export & display
summary_table.to_csv(
    output_path / "catchment_forest_afforestable_summary.csv",
    index=False
)
print("Written:", output_path / "catchment_forest_afforestable_summary.csv")

# In-notebook display
display(summary_table)

In [ ]:

# # ─── 2. Merge the new IDs into your summary ──────────────────────────────────
# table = final_summary.merge(mapping_df, on="HYBAS_ID", how="left")

# # ─── 3. Convert areas from m² to km² ─────────────────────────────────────────
# table["Baseline forest area (km²)"]      = table["forest_flood_equivalent_area"]            / 1e6
# table["Afforestable area (km²)"]         = table["afforestable_including_agriculture_area"] / 1e6

# # ─── 4. Select & rename the columns you need ────────────────────────────────
# summary_table = table[[
#     "new_id",
#     "Baseline forest area (km²)",
#     "forest_flood_equivalent_percentage",
#     "Afforestable area (km²)",
#     "afforestable_including_agriculture_percentage"
# ]].rename(columns={
#     "new_id":                                "Catchment ID",
#     "forest_flood_equivalent_percentage":    "Baseline forest (%)",
#     "afforestable_including_agriculture_percentage": "Afforestable (%)"
# })

# # ─── 5. Sort by the new 1–103 ID ────────────────────────────────────────────
# summary_table = summary_table.sort_values("Catchment ID").reset_index(drop=True)

# # ─── 6. Export &/or display ─────────────────────────────────────────────────
# summary_table.to_csv(output_path / "catchment_forest_afforestable_summary.csv", index=False)
# print("Written:", output_path / "catchment_forest_afforestable_summary.csv")

# # If you want to see it in your notebook
# display(summary_table)

In [ ]:
# mapping_path = output_dir / "catchment_connectivity.csv"

# mapping_df = pd.read_csv(mapping_path, dtype={"HYBAS_ID": str})
# # keep only the two relevant columns
# mapping_df = mapping_df[['HYBAS_ID', 'new_id']]


In [ ]:
# #─── 1. Merge new_id into hydro_summary ────────────────────────────────────────
# # Ensure HYBAS_ID is same dtype in both
# hydro_summary['HYBAS_ID'] = hydro_summary['HYBAS_ID'].astype(str)
# hydro_summary = hydro_summary.merge(
#     mapping_df, on='HYBAS_ID', how='left'
# )
# # Check for any missing
# missing = hydro_summary['new_id'].isna().sum()
# if missing:
#     raise ValueError(f"{missing} catchments failed to match on HYBAS_ID")

# # ─── 2. Propagate new_id into forest_gdf & future_forest_gdf ─────────────────
# forest_gdf['HYBAS_ID'] = forest_gdf['HYBAS_ID'].astype(str)
# forest_gdf = forest_gdf.merge(mapping_df, on='HYBAS_ID', how='left')

# future_forest_gdf['HYBAS_ID'] = future_forest_gdf['HYBAS_ID'].astype(str)
# future_forest_gdf = future_forest_gdf.merge(mapping_df, on='HYBAS_ID', how='left')

# # ─── 3. Now, when you build Figure 1(a) or 1(b), after plotting layers ────────

# def plot_with_labels(gdf, cmap, norm, title):
#     fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

#     # draw patches
#     gdf.plot(ax=ax, color=gdf['color'], linewidth=0, alpha=0.8)

#     # draw catchment boundaries
#     hydro_summary.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5)

#     # colorbar
#     sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm._A = []
#     cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
#     cbar.set_label('% forest cover', fontsize=12, family='Times New Roman')

#     # add new_id labels at representative points
#     for _, row in hydro_summary.iterrows():
#         pt = row.geometry.representative_point()
#         ax.text(
#             pt.x, pt.y,
#             str(int(row.new_id)),
#             fontsize=6, ha='center', va='center',
#             color='black', fontweight='bold'
#         )

#     # legend (just boundaries)
#     handle = Line2D([0], [0], color='black', linewidth=0.5, label='Catchment boundary')
#     ax.legend(handles=[handle], title='Legend',
#               bbox_to_anchor=(0.5, -0.1), loc='upper center',
#               frameon=False, fontsize=12, title_fontsize=14,
#               prop={'family': 'Times New Roman'}).get_title().set_position((0,10))

#     # scale bar & north arrow (reuse your functions)
#     add_scale_bar(ax)
#     add_north_arrow(ax)

#     plt.title(title, fontsize=20, fontweight='bold',
#               fontname='Times New Roman', loc='center', pad=20)
#     ax.set_axis_off()
#     plt.tight_layout()
#     plt.show()

# # ─── 4. Compute shared cmap & norm (from baseline) ────────────────────────────
# cmap = plt.colormaps['Greens']
# vmax_shared = forest_gdf['forest_flood_equivalent_percentage'].max()
# norm = mpl.colors.Normalize(vmin=0, vmax=vmax_shared)

# # recompute colors on forest_gdf & future_forest_gdf
# forest_gdf['color'] = forest_gdf['forest_flood_equivalent_percentage']\
#     .apply(lambda pct: mcolors.to_hex(cmap(norm(pct))))
# future_forest_gdf['color'] = future_forest_gdf['total_future_forest_including_agri_percentage']\
#     .apply(lambda pct: mcolors.to_hex(cmap(norm(pct))))

# # ─── 5. Plot your two figures ─────────────────────────────────────────────────
# plot_with_labels(
#     forest_gdf, cmap, norm,
#     "Figure 1(a) Baseline Forest patches shaded by catchment-% forest"
# )
# plot_with_labels(
#     future_forest_gdf, cmap, norm,
#     "Figure 1(b) Total Future Forest patches shaded by catchment-% potential"
# )

In [ ]:
# # ─── REBUILD hydro_summary ─────────────────────────────────────────────────────
# # Merge both your baseline % and future % into the basin geometries
# hydro_summary = hydrobasins_clipped_saved.merge(
#     final_summary[[
#         'HYBAS_ID',
#         'forest_flood_equivalent_percentage',
#         'total_future_forest_including_agri_percentage'
#     ]],
#     on='HYBAS_ID',
#     how='left'
# ).fillna({
#     'forest_flood_equivalent_percentage': 0,
#     'total_future_forest_including_agri_percentage': 0
# })

# # --- REBUILD forest_gdf so it has forest_flood_equivalent_percentage ---
# forest_gdf = (
#     expanded_gdf[
#         expanded_gdf['LandUseCategory'] == 'forest_flood_equivalent_classes'
#     ]
#     .to_crs(jamaica_metric_grid_crs)
#     .merge(
#         hydro_summary[['HYBAS_ID','forest_flood_equivalent_percentage']],
#         on='HYBAS_ID',
#         how='left'
#     )
#     .fillna({'forest_flood_equivalent_percentage': 0})
# )

# # --- 1. Compute a hex‐colour for each patch from a Greens ramp ---
# cmap = plt.colormaps['Greens']                              # new API
# norm = mpl.colors.Normalize(
#     vmin=forest_gdf['forest_flood_equivalent_percentage'].min(),
#     vmax=forest_gdf['forest_flood_equivalent_percentage'].max()
# )
# forest_gdf['color'] = forest_gdf['forest_flood_equivalent_percentage']\
#     .apply(lambda pct: mcolors.to_hex(cmap(norm(pct))))

In [ ]:
# # --- 1. Compute a hex-color for each patch from a Greens ramp ---
# cmap = plt.cm.get_cmap('Greens')
# norm = mpl.colors.Normalize(
#     vmin=forest_gdf['forest_flood_equivalent_percentage'].min(),
#     vmax=forest_gdf['forest_flood_equivalent_percentage'].max()
# )
# forest_gdf['color'] = forest_gdf['forest_flood_equivalent_percentage']\
#     .apply(lambda pct: mcolors.to_hex(cmap(norm(pct))))

# # --- 2. Define scale bar & north arrow functions (as you had) ---
# def add_scale_bar(ax, length_km=20, location=(0.9, 0.08),
#                   linewidth=2, tick_height=0.01,
#                   label_offset=0.02, km_offset=0.03):
#     x, y = location
#     half = 0.05
#     # bar
#     ax.plot([x-half, x+half], [y, y], transform=ax.transAxes,
#             color='black', linewidth=linewidth)
#     # ticks
#     for pos in (x-half, x, x+half):
#         ax.plot([pos, pos], [y-tick_height/2, y+tick_height/2],
#                 transform=ax.transAxes, color='black', linewidth=linewidth)
#     # labels
#     ax.text(x-half, y-tick_height-label_offset, "0",
#             transform=ax.transAxes, ha='center', va='center', fontsize=10)
#     ax.text(x, y-tick_height-label_offset, f"{length_km//2}",
#             transform=ax.transAxes, ha='center', va='center', fontsize=10)
#     ax.text(x+half, y-tick_height-label_offset, f"{length_km}",
#             transform=ax.transAxes, ha='center', va='center', fontsize=10)
#     ax.text(x+half+km_offset, y, "km",
#             transform=ax.transAxes, ha='left', va='center', fontsize=12)

# def add_north_arrow(ax, location=(0.9, 0.15), size=0.05,
#                     fontsize=12, label_offset=0.03):
#     x, y = location
#     ax.annotate(
#         '', xy=(x, y+size), xycoords='axes fraction',
#         xytext=(x, y), textcoords='axes fraction',
#         arrowprops=dict(facecolor='black', edgecolor='black',
#                         headwidth=10, headlength=15, width=5)
#     )
#     ax.text(x, y+size+label_offset, "N", transform=ax.transAxes,
#             fontsize=fontsize, fontweight="bold",
#             ha="center", va="center", color="black")

# # --- 3. Build the figure ---
# fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# # 3a) Forest patches, colored by catchment %  
# forest_gdf.plot(
#     ax=ax,
#     color=forest_gdf['color'],
#     linewidth=0,
#     alpha=0.8,
#     label='Forest patches'
# )

# # 3b) Catchment boundaries on top  
# hydro_summary.boundary.plot(
#     ax=ax,
#     edgecolor='black',
#     linewidth=0.5,
#     label='Catchment boundary'
# )

# # 3c) Continuous colorbar for the % ramp  
# sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
# sm._A = []
# cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
# cbar.set_label('Forest cover (% of catchment)', fontsize=12, family='Times New Roman')

# # 3d) Legend for the two main layers  
# handles = [
#     mpatches.Patch(color=forest_gdf['color'].iloc[-1], label='Forest patches'),
#     Line2D([0], [0], color='black', linewidth=0.5, label='Catchment boundaries')
# ]
# legend = ax.legend(
#     handles=handles,
#     title='Legend',
#     bbox_to_anchor=(0.5, -0.1),
#     loc='upper center',
#     ncol=2,
#     frameon=False,
#     fontsize=12,
#     title_fontsize=14,
#     prop={'family': 'Times New Roman'}
# )
# legend.get_title().set_position((0, 10))

# # 3e) Add scale bar & north arrow  
# add_scale_bar(ax, length_km=20)
# add_north_arrow(ax)

# # 3f) Title & styling  
# plt.title(
#     "Figure 1(a) Baseline Forest area shaded by catchment-level % Cover",
#     fontsize=20,
#     fontweight='bold',
#     fontname='Times New Roman',
#     loc='center',
#     pad=20
# )
# ax.set_axis_off()
# plt.tight_layout()
# plt.show()

In [ ]:
# # ----------------------------------------------------------------------------
# # ❗️ STEP 0: Rebuild hydro_summary so it has the total_future_forest... column
# # ----------------------------------------------------------------------------
# # Assume you still have in memory:
# #   hydrobasins_clipped_saved  : GeoDataFrame of your clipped basins
# #   final_summary              : DataFrame with HYBAS_ID and total_future_forest_including_agri_percentage

# hydro_summary = hydrobasins_clipped_saved.merge(
#     final_summary[
#         ['HYBAS_ID', 'total_future_forest_including_agri_percentage']
#     ],
#     on='HYBAS_ID',
#     how='left'
# ).fillna({'total_future_forest_including_agri_percentage': 0})

In [ ]:
# # --- 1. Define a shared “Greens” ramp & Normalize to baseline max ---
# cmap = plt.colormaps['Greens']
# vmax_shared = forest_gdf['forest_flood_equivalent_percentage'].max()
# norm = mpl.colors.Normalize(vmin=0, vmax=vmax_shared)

# # --- 2. Rebuild hydro_summary so it has the future‐forest % field ---
# hydro_summary = hydrobasins_clipped_saved.merge(
#     final_summary[['HYBAS_ID','total_future_forest_including_agri_percentage']],
#     on='HYBAS_ID',
#     how='left'
# ).fillna({'total_future_forest_including_agri_percentage': 0})

# # --- 3. Build future_forest_gdf with that % merged in ---
# future_forest_gdf = (
#     expanded_gdf[
#         expanded_gdf['LandUseCategory'].isin([
#             'forest_flood_equivalent_classes',
#             'afforestable_including_agriculture'
#         ])
#     ]
#     .to_crs(jamaica_metric_grid_crs)
#     .merge(
#         hydro_summary[['HYBAS_ID','total_future_forest_including_agri_percentage']],
#         on='HYBAS_ID',
#         how='left'
#     )
#     .fillna({'total_future_forest_including_agri_percentage': 0})
# )

# # --- 4. Compute a hex‐color for each patch using the shared ramp & norm ---
# future_forest_gdf['color'] = (
#     future_forest_gdf['total_future_forest_including_agri_percentage']
#     .apply(lambda pct: mcolors.to_hex(cmap(norm(pct))))
# )

# # --- 5. Define scale bar & north arrow helpers ---
# def add_scale_bar(ax, length_km=20, location=(0.9, 0.08),
#                   linewidth=2, tick_height=0.01,
#                   label_offset=0.02, km_offset=0.03):
#     x, y = location
#     half = 0.05
#     ax.plot([x-half, x+half], [y, y], transform=ax.transAxes,
#             color='black', linewidth=linewidth)
#     for pos in (x-half, x, x+half):
#         ax.plot([pos, pos], [y-tick_height/2, y+tick_height/2],
#                 transform=ax.transAxes, color='black', linewidth=linewidth)
#     ax.text(x-half, y-tick_height-label_offset, "0",
#             transform=ax.transAxes, ha='center', va='center', fontsize=10)
#     ax.text(x, y-tick_height-label_offset, f"{length_km//2}",
#             transform=ax.transAxes, ha='center', va='center', fontsize=10)
#     ax.text(x+half, y-tick_height-label_offset, f"{length_km}",
#             transform=ax.transAxes, ha='center', va='center', fontsize=10)
#     ax.text(x+half+km_offset, y, "km",
#             transform=ax.transAxes, ha='left', va='center', fontsize=12)

# def add_north_arrow(ax, location=(0.9, 0.15), size=0.05,
#                     fontsize=12, label_offset=0.03):
#     x, y = location
#     ax.annotate(
#         '', xy=(x, y+size), xycoords='axes fraction',
#         xytext=(x, y), textcoords='axes fraction',
#         arrowprops=dict(facecolor='black', edgecolor='black',
#                         headwidth=10, headlength=15, width=5)
#     )
#     ax.text(x, y+size+label_offset, "N", transform=ax.transAxes,
#             fontsize=fontsize, fontweight="bold",
#             ha="center", va="center", color="black")

# # --- 6. Build Figure 1(b) using the shared scale & ramp ---
# fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# # a) Future forest patches only, colored by shared ramp
# future_forest_gdf.plot(
#     ax=ax,
#     color=future_forest_gdf['color'],
#     linewidth=0,
#     alpha=0.8,
#     label='Future forest patches'
# )

# # b) Overlay catchment boundaries
# hydro_summary.boundary.plot(
#     ax=ax,
#     edgecolor='black',
#     linewidth=0.5,
#     label='Catchment boundary'
# )

# # c) Shared continuous colorbar
# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm._A = []
# cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
# cbar.set_label(
#     'Forest cover (% of catchment)',
#     fontsize=12,
#     family='Times New Roman'
# )

# # d) Minimal legend
# handles = [
#     mpatches.Patch(color='white', label=''),  # placeholder to align
#     Line2D([0], [0], color='black', linewidth=0.5, label='Catchment boundaries')
# ]
# legend = ax.legend(
#     handles=handles,
#     title='Legend',
#     bbox_to_anchor=(0.5, -0.1),
#     loc='upper center',
#     ncol=1,
#     frameon=False,
#     fontsize=12,
#     title_fontsize=14,
#     prop={'family': 'Times New Roman'}
# )
# legend.get_title().set_position((0, 10))

# # e) Add scale bar & north arrow
# add_scale_bar(ax)
# add_north_arrow(ax)

# # f) Title & layout
# plt.title(
#     "Figure 1(b) Total Future Forest Potential\nshaded by catchment-level % cover",
#     fontsize=20,
#     fontweight='bold',
#     fontname='Times New Roman',
#     loc='center',
#     pad=20
# )
# ax.set_axis_off()
# plt.tight_layout()
# plt.show()